# Data.gov.hk Web Crawling and API Access Starter Notebook

## 🎯 Project Objective
This notebook helps you explore data.gov.hk to find datasets suitable for:
- **Regression analysis** or **simulation modeling**
- Connection to specific **government policy or decision**
- **Data governance improvement** recommendations

## 📋 Team Requirements
Your team must:
1. **Be patient** - systematically explore multiple datasets
2. **Select one specific dataset** connected to government policy
3. **Choose modeling approach:**
   - If data supports it: regression/simulation analysis
   - If descriptive only: data governance improvement recommendations

## 🌟 Why This Topic?
- **More versatile and open-ended** than other topics
- **Maximum flexibility** for creative exploration
- **Exciting with some uncertainty** - perfect for adventurous teams!
- **No prior foundation** - you're building something completely new

## Please note that the codes and instructions are generated by AI and could be wrong. Please consult AI to revise and adapt and seek help from your human teachers if needed. 

## 🛠️ Setup: Install Required Libraries

In [3]:
# Install required packages
!pip install requests beautifulsoup4 pandas matplotlib seaborn plotly lxml openpyxl


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip


## 📚 Import Libraries

In [2]:
import requests
import pandas as pd
import json
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime, timedelta
import time
import warnings
warnings.filterwarnings('ignore')

# Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✅ All libraries imported successfully!")
print(f"📅 Notebook initialized on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ All libraries imported successfully!
📅 Notebook initialized on: 2025-09-16 00:07:41


# 🔍 Phase 1: Discover Available Datasets

## Step 1: Explore Data.gov.hk Categories

In [4]:
# Function to scrape data.gov.hk main categories
def explore_data_gov_categories():
    """
    Scrape the main categories from data.gov.hk
    """
    base_url = "https://data.gov.hk/en/"
    
    try:
        response = requests.get(base_url, timeout=10)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.content, 'html.parser')
        print("🌐 Successfully connected to data.gov.hk!")
        print(f"📄 Page title: {soup.title.string if soup.title else 'No title found'}")
        
        # Look for category links or sections
        categories = []
        
        # Try to find navigation or category sections
        nav_links = soup.find_all('a', href=True)
        for link in nav_links[:20]:  # Limit to first 20 for exploration
            if 'dataset' in link.get('href', '').lower():
                categories.append({
                    'text': link.get_text(strip=True),
                    'href': link.get('href')
                })
        
        return categories
        
    except requests.RequestException as e:
        print(f"❌ Error connecting to data.gov.hk: {e}")
        return []

# Explore categories
categories = explore_data_gov_categories()
print(f"\n📊 Found {len(categories)} potential category links:")
for i, cat in enumerate(categories[:10], 1):
    print(f"{i}. {cat['text'][:50]}... -> {cat['href'][:50]}...")

🌐 Successfully connected to data.gov.hk!
📄 Page title: Home | DATA.GOV.HK

📊 Found 9 potential category links:
1. Datasets... -> /en-datasets...
2. City Management and Utilities... -> /en-datasets/category/city-management...
3. Climate and Weather... -> /en-datasets/category/climate-and-weather...
4. Commerce and Industry... -> /en-datasets/category/commerce-and-industry...
5. Development, Geography and Land Information... -> /en-datasets/category/development...
6. Education... -> /en-datasets/category/education...
7. Employment and Labour... -> /en-datasets/category/employment-and-labour...
8. Environment... -> /en-datasets/category/environment...
9. Finance... -> /en-datasets/category/finance...


## Step 2: Access Data.gov.hk API

In [5]:
# Step 1: Extract Data Providers from data.gov.hk
def extract_data_providers():
    """
    Extract all data providers from https://data.gov.hk/en/providers
    """
    providers_url = "https://data.gov.hk/en/providers"
    
    try:
        print(f"🔍 Scraping data providers from: {providers_url}")
        response = requests.get(providers_url, timeout=15)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.content, 'html.parser')
        print("✅ Successfully connected to data.gov.hk providers page!")
        
        providers = []
        
        # Look for provider links - they typically follow the pattern /en-datasets/provider/[provider-id]
        provider_links = soup.find_all('a', href=True)
        
        for link in provider_links:
            href = link.get('href', '')
            if '/en-datasets/provider/' in href:
                provider_name = link.get_text(strip=True)
                provider_id = href.split('/provider/')[-1]
                
                if provider_name and provider_id:
                    providers.append({
                        'name': provider_name,
                        'id': provider_id,
                        'url': f"https://data.gov.hk{href}" if href.startswith('/') else href
                    })
        
        # Remove duplicates based on provider_id
        unique_providers = {}
        for provider in providers:
            if provider['id'] not in unique_providers:
                unique_providers[provider['id']] = provider
        
        providers_list = list(unique_providers.values())
        
        print(f"\n📊 Found {len(providers_list)} data providers:")
        for i, provider in enumerate(providers_list[:10], 1):
            print(f"{i:2d}. {provider['name']} (ID: {provider['id']})")
        
        if len(providers_list) > 10:
            print(f"    ... and {len(providers_list) - 10} more providers")
        
        return providers_list
        
    except requests.RequestException as e:
        print(f"❌ Error connecting to providers page: {e}")
        return []
    except Exception as e:
        print(f"❌ Error parsing providers: {e}")
        return []

# Extract providers
providers = extract_data_providers()

🔍 Scraping data providers from: https://data.gov.hk/en/providers
✅ Successfully connected to data.gov.hk providers page!

📊 Found 0 data providers:


# Step 2: Explore Specific Provider Datasets

Now let's explore datasets from specific providers. We'll use the Buildings Department (hk-bd) as an example:

In [ ]:
# Step 2: Explore datasets from a specific provider
def explore_provider_datasets(provider_id, provider_name=None):
    """
    Extract all datasets from a specific provider page
    Example: provider_id = 'hk-bd' for Buildings Department
    """
    provider_url = f"https://data.gov.hk/en-datasets/provider/{provider_id}"
    
    try:
        print(f"🔍 Exploring datasets from provider: {provider_id}")
        print(f"🌐 URL: {provider_url}")
        
        response = requests.get(provider_url, timeout=15)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Get provider name from page title or header
        if not provider_name:
            title = soup.find('title')
            if title:
                provider_name = title.get_text().split('-')[0].strip()
            else:
                provider_name = provider_id
        
        print(f"✅ Successfully connected to {provider_name} datasets page!")
        
        datasets = []
        
        # Look for dataset links - they typically follow pattern /en-dataset/[dataset-id]/[provider-id]/[dataset-name]
        dataset_links = soup.find_all('a', href=True)
        
        for link in dataset_links:
            href = link.get('href', '')
            if '/en-dataset/' in href and provider_id in href:
                dataset_title = link.get_text(strip=True)
                
                # Extract dataset ID from URL
                url_parts = href.split('/')
                if len(url_parts) >= 3:
                    dataset_id = url_parts[2] if url_parts[1] == 'en-dataset' else None
                    
                    if dataset_title and dataset_id:
                        datasets.append({
                            'title': dataset_title,
                            'id': dataset_id,
                            'url': f"https://data.gov.hk{href}" if href.startswith('/') else href,
                            'provider': provider_name,
                            'provider_id': provider_id
                        })
        
        # Remove duplicates based on dataset_id
        unique_datasets = {}
        for dataset in datasets:
            if dataset['id'] not in unique_datasets:
                unique_datasets[dataset['id']] = dataset
        
        datasets_list = list(unique_datasets.values())
        
        print(f"\n📊 Found {len(datasets_list)} datasets from {provider_name}:")
        for i, dataset in enumerate(datasets_list[:10], 1):
            print(f"{i:2d}. {dataset['title'][:60]}...")
            print(f"    ID: {dataset['id']}")
            print(f"    URL: {dataset['url']}")
            print()
        
        if len(datasets_list) > 10:
            print(f"    ... and {len(datasets_list) - 10} more datasets")
        
        return datasets_list
        
    except requests.RequestException as e:
        print(f"❌ Error connecting to provider page: {e}")
        return []
    except Exception as e:
        print(f"❌ Error parsing datasets: {e}")
        return []

# Example: Explore Buildings Department datasets
print("? EXPLORING BUILDINGS DEPARTMENT DATASETS")
print("=" * 60)
bd_datasets = explore_provider_datasets('hk-bd', 'Buildings Department')

print("\n" + "=" * 60)
print("? TRY OTHER PROVIDERS:")
print("• Transport Department: 'hk-td'")
print("• Environmental Protection Department: 'hk-epd'") 
print("• Census and Statistics Department: 'hk-censtatd'")
print("• Health Department: 'hk-dh'")
print("\nUsage: datasets = explore_provider_datasets('provider-id')")

# 🔬 Phase 2: Dataset Access Methods

## Step 3: Explore Different Data Access Methods

Let's explore how to access data from a specific dataset using different methods (API, CSV, JSON, etc.):

In [ ]:
# Step 3: Explore data access methods for a specific dataset
def explore_dataset_access_methods(dataset_url):
    """
    Explore different ways to access data from a specific dataset page
    """
    try:
        print(f"🔍 Exploring data access methods for:")
        print(f"🌐 {dataset_url}")
        
        response = requests.get(dataset_url, timeout=15)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Get dataset title
        title_tag = soup.find('h1') or soup.find('title')
        dataset_title = title_tag.get_text(strip=True) if title_tag else "Unknown Dataset"
        
        print(f"📊 Dataset: {dataset_title}")
        print("=" * 80)
        
        access_methods = []
        
        # Look for different download/access options
        download_links = soup.find_all('a', href=True)
        
        for link in download_links:
            href = link.get('href', '')
            link_text = link.get_text(strip=True).lower()
            
            # Check for API endpoints
            if any(keyword in href.lower() for keyword in ['api', 'json', 'xml']):
                access_methods.append({
                    'type': 'API/JSON',
                    'url': href if href.startswith('http') else f"https://data.gov.hk{href}",
                    'description': link.get_text(strip=True),
                    'format': 'JSON/XML'
                })
            
            # Check for CSV files
            elif href.endswith('.csv') or 'csv' in link_text:
                access_methods.append({
                    'type': 'CSV File',
                    'url': href if href.startswith('http') else f"https://data.gov.hk{href}",
                    'description': link.get_text(strip=True),
                    'format': 'CSV'
                })
            
            # Check for Excel files
            elif href.endswith(('.xlsx', '.xls')) or 'excel' in link_text:
                access_methods.append({
                    'type': 'Excel File',
                    'url': href if href.startswith('http') else f"https://data.gov.hk{href}",
                    'description': link.get_text(strip=True),
                    'format': 'Excel'
                })
            
            # Check for JSON files
            elif href.endswith('.json') or 'json' in link_text:
                access_methods.append({
                    'type': 'JSON File',
                    'url': href if href.startswith('http') else f"https://data.gov.hk{href}",
                    'description': link.get_text(strip=True),
                    'format': 'JSON'
                })
        
        # Remove duplicates based on URL
        unique_methods = {}
        for method in access_methods:
            url_key = method['url']
            if url_key not in unique_methods:
                unique_methods[url_key] = method
        
        methods_list = list(unique_methods.values())
        
        if methods_list:
            print(f"📥 Found {len(methods_list)} data access methods:")
            print()
            
            for i, method in enumerate(methods_list, 1):
                print(f"{i}. {method['type']} ({method['format']})")
                print(f"   📝 Description: {method['description']}")
                print(f"   🔗 URL: {method['url']}")
                print()
        else:
            print("⚠️ No clear data access methods found on this page.")
            print("💡 Try looking for 'Download', 'API', 'JSON', or 'CSV' links manually")
        
        # Look for additional information
        print("📋 ADDITIONAL DATASET INFORMATION:")
        
        # Look for update frequency
        text_content = soup.get_text().lower()
        if 'daily' in text_content:
            print("🔄 Update Frequency: Likely Daily")
        elif 'monthly' in text_content:
            print("🔄 Update Frequency: Likely Monthly")
        elif 'annual' in text_content:
            print("🔄 Update Frequency: Likely Annual")
        else:
            print("🔄 Update Frequency: Not clearly specified")
        
        # Look for data size information
        if 'mb' in text_content or 'megabyte' in text_content:
            print("📏 Data Size: Medium (MB range)")
        elif 'gb' in text_content or 'gigabyte' in text_content:
            print("📏 Data Size: Large (GB range)")
        elif 'kb' in text_content or 'kilobyte' in text_content:
            print("? Data Size: Small (KB range)")
        else:
            print("📏 Data Size: Not specified")
        
        return {
            'title': dataset_title,
            'url': dataset_url,
            'access_methods': methods_list,
            'total_methods': len(methods_list)
        }
        
    except requests.RequestException as e:
        print(f"❌ Error connecting to dataset page: {e}")
        return None
    except Exception as e:
        print(f"❌ Error parsing dataset page: {e}")
        return None

# Example usage instructions
print("🎯 DATASET ACCESS METHOD EXPLORER")
print("=" * 50)
print("Usage: dataset_info = explore_dataset_access_methods('dataset_url')")
print("\n? How to find dataset URLs:")
print("1. Use explore_provider_datasets() to get dataset URLs")
print("2. Or manually browse https://data.gov.hk/en-datasets/provider/[provider-id]")
print("3. Click on any dataset to get its URL")
print("\n📝 Example:")
print("dataset_info = explore_dataset_access_methods(bd_datasets[0]['url'])")

# Step 4: Smart Data Access and Preview

Now let's create a smart function that can automatically access data using the best available method:

In [ ]:
# Step 4: Smart data access function
def smart_dataset_access(dataset_url, sample_size=1000):
    """
    Intelligently access dataset using the best available method
    Automatically tries different formats and provides preview
    """
    print(f"🤖 SMART DATASET ACCESS")
    print("=" * 50)
    
    # First, explore available access methods
    dataset_info = explore_dataset_access_methods(dataset_url)
    
    if not dataset_info or not dataset_info['access_methods']:
        print("❌ No data access methods found")
        return None, None
    
    print(f"\n🎯 Attempting to access data using {len(dataset_info['access_methods'])} available methods...")
    
    # Try each access method in order of preference: API/JSON > CSV > Excel
    method_priority = ['API/JSON', 'JSON File', 'CSV File', 'Excel File']
    
    for priority_type in method_priority:
        for method in dataset_info['access_methods']:
            if method['type'] == priority_type:
                try:
                    print(f"\n? Trying {method['type']}: {method['url']}")
                    
                    df = None
                    file_type = method['format']
                    
                    if method['type'] in ['API/JSON', 'JSON File']:
                        response = requests.get(method['url'], timeout=30)
                        response.raise_for_status()
                        data = response.json()
                        
                        if isinstance(data, list):
                            df = pd.DataFrame(data)
                        elif isinstance(data, dict):
                            # Try to find the actual data within the JSON structure
                            for key, value in data.items():
                                if isinstance(value, list) and len(value) > 0:
                                    df = pd.DataFrame(value)
                                    break
                            if df is None:
                                df = pd.json_normalize(data)
                        else:
                            print(f"⚠️ Unexpected JSON structure: {type(data)}")
                            continue
                    
                    elif method['type'] == 'CSV File':
                        df = pd.read_csv(method['url'])
                    
                    elif method['type'] == 'Excel File':
                        df = pd.read_excel(method['url'])
                    
                    if df is not None and len(df) > 0:
                        print(f"✅ Successfully accessed data using {method['type']}!")
                        
                        # Create comprehensive report
                        report = create_dataset_report(df, dataset_info['title'], method)
                        
                        # Return sample if dataset is large
                        if len(df) > sample_size:
                            print(f"📊 Dataset is large ({len(df):,} rows). Returning sample of {sample_size:,} rows.")
                            df_sample = df.sample(n=sample_size, random_state=42)
                            return df_sample, report
                        else:
                            return df, report
                    
                except Exception as e:
                    print(f"❌ Failed to access via {method['type']}: {e}")
                    continue
    
    print("❌ Could not access data using any available method")
    return None, None

def create_dataset_report(df, dataset_title, access_method):
    """
    Create comprehensive dataset report
    """
    report = {
        'dataset_title': dataset_title,
        'access_method': access_method,
        'shape': df.shape,
        'columns': list(df.columns),
        'numeric_columns': list(df.select_dtypes(include=['number']).columns),
        'datetime_columns': [],
        'categorical_columns': list(df.select_dtypes(include=['object', 'category']).columns),
        'missing_data': df.isnull().sum().sum(),
        'sample_data': df.head()
    }
    
    # Detect datetime columns
    for col in df.columns:
        if df[col].dtype == 'datetime64[ns]':
            report['datetime_columns'].append(col)
        elif df[col].dtype == 'object':
            # Try to parse as datetime
            try:
                pd.to_datetime(df[col].dropna().iloc[:100], errors='raise')
                report['datetime_columns'].append(col)
            except:
                pass
    
    # Display comprehensive report
    print(f"\n📊 COMPREHENSIVE DATASET REPORT")
    print("=" * 60)
    print(f"📁 Dataset: {report['dataset_title']}")
    print(f"? Access Method: {report['access_method']['type']}")
    print(f"📄 Format: {report['access_method']['format']}")
    print(f"📏 Shape: {report['shape'][0]:,} rows × {report['shape'][1]} columns")
    print(f"💾 Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
    print(f"\n📊 COLUMN ANALYSIS:")
    print(f"🔢 Numeric columns: {len(report['numeric_columns'])}")
    print(f"📅 DateTime columns: {len(report['datetime_columns'])}")
    print(f"? Categorical columns: {len(report['categorical_columns'])}")
    print(f"❓ Missing values: {report['missing_data']:,}")
    
    if report['numeric_columns']:
        print(f"\n? Numeric columns: {', '.join(report['numeric_columns'][:5])}{'...' if len(report['numeric_columns']) > 5 else ''}")
    
    if report['datetime_columns']:
        print(f"? DateTime columns: {', '.join(report['datetime_columns'])}")
    
    print(f"\n📋 Sample Data:")
    display(report['sample_data'])
    
    # Modeling potential assessment
    modeling_score = 0
    if report['shape'][0] > 100: modeling_score += 1
    if report['shape'][0] > 1000: modeling_score += 1
    if len(report['numeric_columns']) >= 2: modeling_score += 1
    if len(report['datetime_columns']) >= 1: modeling_score += 1
    if report['missing_data'] / (report['shape'][0] * report['shape'][1]) < 0.1: modeling_score += 1
    
    print(f"\n⭐ MODELING POTENTIAL SCORE: {modeling_score}/5")
    
    if modeling_score >= 4:
        print("🎯 HIGH POTENTIAL for regression/simulation analysis!")
        print("✅ Recommended approach: Quantitative modeling")
    elif modeling_score >= 3:
        print("⚡ MODERATE POTENTIAL - consider mixed approach")
        print("🔄 Recommended approach: Basic modeling + governance analysis")
    else:
        print("🔧 FOCUS ON DATA GOVERNANCE improvements")
        print("📋 Recommended approach: Data governance improvement analysis")
    
    return report

# Usage example
print("? SMART DATASET ACCESS")
print("=" * 40)
print("This function automatically finds and accesses data using the best available method")
print("\n? Usage:")
print("df, report = smart_dataset_access(dataset_url)")
print("\n💡 To get dataset URLs, first run:")
print("datasets = explore_provider_datasets('provider-id')")
print("Then use: df, report = smart_dataset_access(datasets[0]['url'])")

# 📊 Phase 3: Complete Workflow Example

## Step 5: Complete Data Exploration Workflow

Let's demonstrate the complete workflow from provider discovery to data analysis:

In [ ]:
# Step 5: Complete workflow demonstration
def complete_exploration_workflow(provider_id, max_datasets=3):
    """
    Demonstrate complete workflow from provider to data analysis
    """
    print("? COMPLETE DATA EXPLORATION WORKFLOW")
    print("=" * 80)
    
    # Step 1: Get all providers (if needed)
    print("1️⃣ DISCOVERING PROVIDERS...")
    if provider_id not in ['hk-bd', 'hk-td', 'hk-epd', 'hk-censtatd', 'hk-dh']:
        providers = extract_data_providers()
        if providers:
            print(f"   Found {len(providers)} providers. Use any provider_id from the list above.")
    
    # Step 2: Explore provider datasets
    print(f"\n2️⃣ EXPLORING DATASETS FROM PROVIDER: {provider_id}")
    datasets = explore_provider_datasets(provider_id)
    
    if not datasets:
        print("❌ No datasets found for this provider")
        return
    
    print(f"   Found {len(datasets)} datasets")
    
    # Step 3: Try to access data from first few datasets
    print(f"\n3️⃣ ATTEMPTING DATA ACCESS (trying first {max_datasets} datasets):")
    
    successful_datasets = []
    
    for i, dataset in enumerate(datasets[:max_datasets]):
        print(f"\n📊 Dataset {i+1}: {dataset['title'][:50]}...")
        print("-" * 60)
        
        try:
            df, report = smart_dataset_access(dataset['url'], sample_size=500)
            
            if df is not None and report is not None:
                successful_datasets.append({
                    'dataset': dataset,
                    'dataframe': df,
                    'report': report
                })
                print(f"✅ Successfully accessed dataset {i+1}")
            else:
                print(f"❌ Could not access dataset {i+1}")
                
        except Exception as e:
            print(f"❌ Error with dataset {i+1}: {e}")
    
    # Step 4: Summary and recommendations
    print(f"\n4️⃣ WORKFLOW SUMMARY")
    print("=" * 60)
    print(f"📊 Provider: {provider_id}")
    print(f"📁 Total datasets found: {len(datasets)}")
    print(f"✅ Successfully accessed: {len(successful_datasets)}")
    print(f"❌ Failed to access: {max_datasets - len(successful_datasets)}")
    
    if successful_datasets:
        print(f"\n🎯 RECOMMENDATIONS FOR FURTHER ANALYSIS:")
        
        for i, item in enumerate(successful_datasets, 1):
            report = item['report']
            dataset = item['dataset']
            
            print(f"\n{i}. {dataset['title'][:50]}...")
            print(f"   📏 Size: {report['shape'][0]:,} rows × {report['shape'][1]} columns")
            print(f"   🔢 Numeric columns: {len(report['numeric_columns'])}")
            print(f"   📅 DateTime columns: {len(report['datetime_columns'])}")
            
            # Quick recommendation
            if len(report['numeric_columns']) >= 3 and report['shape'][0] > 1000:
                print(f"   🎯 HIGH POTENTIAL for regression/simulation analysis")
            elif len(report['numeric_columns']) >= 1 and report['shape'][0] > 100:
                print(f"   ⚡ MODERATE POTENTIAL for quantitative analysis")
            else:
                print(f"   🔧 FOCUS ON data governance improvement analysis")
    
    print(f"\n💡 NEXT STEPS:")
    print("1. Choose one dataset with highest modeling potential")
    print("2. Run comprehensive_dataset_analysis() for detailed exploration")
    print("3. Use policy_connection_analysis() to link to government decisions")
    print("4. Complete data_governance_assessment() if taking governance approach")
    
    return successful_datasets

# Quick exploration of different providers
print("🌟 QUICK PROVIDER EXPLORATION")
print("=" * 50)
print("Run complete workflow for different providers:")
print("\n📋 Available providers to try:")
providers_to_try = {
    'hk-bd': 'Buildings Department (Construction, Building Safety)',
    'hk-td': 'Transport Department (Traffic, Public Transport)',
    'hk-epd': 'Environmental Protection Department (Air Quality, Waste)',
    'hk-censtatd': 'Census and Statistics Department (Demographics, Economy)',
    'hk-dh': 'Department of Health (Public Health, Disease Surveillance)'
}

for provider_id, description in providers_to_try.items():
    print(f"• {provider_id}: {description}")

print(f"\n📝 Usage:")
print("results = complete_exploration_workflow('hk-bd')  # Buildings Department")
print("results = complete_exploration_workflow('hk-td')  # Transport Department")
print("results = complete_exploration_workflow('hk-epd')  # Environmental Protection")

## Step 6: Systematic Dataset Categories with Real Provider Examples

Now let's map the theoretical categories to actual data.gov.hk providers:

In [ ]:
# Updated systematic exploration with real data.gov.hk providers
dataset_categories_with_providers = {
    "🚌 Transport": {
        "description": "Traffic patterns, public transport, road safety",
        "policy_connection": "Transport policy, infrastructure planning",
        "modeling_potential": "High - time series, regression analysis",
        "main_provider": "hk-td (Transport Department)",
        "other_providers": ["hk-hyd (Highways Department)", "hk-mtr (MTR Corporation)"],
        "example_datasets": ["Traffic speed data", "Bus route statistics", "Road accident records", "Parking meter usage"],
        "workflow_example": "complete_exploration_workflow('hk-td')"
    },
    "🌱 Environment": {
        "description": "Air quality, waste management, energy",
        "policy_connection": "Environmental protection, sustainability",
        "modeling_potential": "High - correlation analysis, trend prediction",
        "main_provider": "hk-epd (Environmental Protection Department)",
        "other_providers": ["hk-hko (Hong Kong Observatory)", "hk-devb (Development Bureau)"],
        "example_datasets": ["Air quality index", "Waste collection data", "Energy consumption", "Weather monitoring"],
        "workflow_example": "complete_exploration_workflow('hk-epd')"
    },
    "🏥 Health": {
        "description": "Disease surveillance, healthcare utilization",
        "policy_connection": "Public health policy, resource allocation",
        "modeling_potential": "Medium-High - epidemiological modeling",
        "main_provider": "hk-dh (Department of Health)",
        "other_providers": ["hk-ha (Hospital Authority)", "hk-fehd (Food and Environmental Hygiene Department)"],
        "example_datasets": ["Disease surveillance", "Hospital statistics", "Health indicators", "Food safety data"],
        "workflow_example": "complete_exploration_workflow('hk-dh')"
    },
    "🎓 Education": {
        "description": "School performance, enrollment, resources",
        "policy_connection": "Education policy, funding allocation",
        "modeling_potential": "Medium - performance analysis",
        "main_provider": "hk-edb (Education Bureau)",
        "other_providers": ["hk-ugc (University Grants Committee)"],
        "example_datasets": ["School enrollment", "Academic performance", "Education expenditure", "Teacher statistics"],
        "workflow_example": "complete_exploration_workflow('hk-edb')"
    },
    "🏠 Housing & Construction": {
        "description": "Property data, construction permits, building safety",
        "policy_connection": "Housing policy, urban planning, building regulations",
        "modeling_potential": "High - price prediction, safety analysis",
        "main_provider": "hk-bd (Buildings Department)",
        "other_providers": ["hk-hd (Housing Department)", "hk-rvd (Rating and Valuation Department)"],
        "example_datasets": ["Building permits", "Construction records", "Property transactions", "Public housing data"],
        "workflow_example": "complete_exploration_workflow('hk-bd')"
    },
    "💼 Economy & Demographics": {
        "description": "Economic indicators, population statistics, business data",
        "policy_connection": "Economic development, demographic planning",
        "modeling_potential": "High - economic modeling, trend analysis",
        "main_provider": "hk-censtatd (Census and Statistics Department)",
        "other_providers": ["hk-tid (Trade and Industry Department)", "hk-labour (Labour Department)"],
        "example_datasets": ["GDP statistics", "Population census", "Employment data", "Business registrations"],
        "workflow_example": "complete_exploration_workflow('hk-censtatd')"
    }
}

# Display updated categories with real provider information
print("🎯 SYSTEMATIC EXPLORATION WITH REAL DATA.GOV.HK PROVIDERS")
print("=" * 80)

for category, info in dataset_categories_with_providers.items():
    print(f"\n{category}")
    print(f"📝 Description: {info['description']}")
    print(f"🏛️ Policy Connection: {info['policy_connection']}")
    print(f"📊 Modeling Potential: {info['modeling_potential']}")
    print(f"🔧 Main Provider: {info['main_provider']}")
    if info['other_providers']:
        print(f"🔗 Other Providers: {', '.join(info['other_providers'])}")
    print(f"💡 Example Datasets: {', '.join(info['example_datasets'][:2])}...")
    print(f"🚀 Try It: {info['workflow_example']}")
    print("-" * 60)

print("\n💡 SYSTEMATIC EXPLORATION STRATEGY:")
print("1. Choose a category that interests you")
print("2. Run the workflow example for that category's main provider")
print("3. Examine the successful datasets for modeling potential")
print("4. Select ONE dataset for your project focus")
print("5. Use the analysis functions to explore policy connections and governance")

## Step 7: Data Governance Assessment Framework

Now that you have working tools to explore data.gov.hk, here's how to connect your data exploration to governance analysis:

In [ ]:
def governance_assessment_framework(provider_id, dataset_results):
    """
    Assess data governance practices based on exploration results
    
    Args:
        provider_id: The government department code (e.g., 'hk-bd')
        dataset_results: Results from explore_provider_datasets()
    
    Returns:
        dict: Governance assessment scoring
    """
    
    if not dataset_results or not dataset_results.get('datasets'):
        print(f"❌ No datasets found for {provider_id}")
        return None
    
    datasets = dataset_results['datasets']
    total_datasets = len(datasets)
    
    # Data Quality Indicators
    has_api_access = sum(1 for d in datasets if 'API' in str(d.get('access_methods', [])))
    has_machine_readable = sum(1 for d in datasets if any(fmt in str(d.get('access_methods', [])) for fmt in ['JSON', 'CSV', 'XML']))
    has_metadata = sum(1 for d in datasets if d.get('description') and len(d.get('description', '')) > 50)
    
    # Update Frequency Assessment
    update_indicators = []
    for dataset in datasets:
        desc = str(dataset.get('description', '')).lower()
        if any(word in desc for word in ['daily', 'weekly', 'monthly', 'real-time', 'live']):
            update_indicators.append('regular')
        elif any(word in desc for word in ['annual', 'yearly', 'census']):
            update_indicators.append('periodic')
        else:
            update_indicators.append('unclear')
    
    # Calculate Governance Scores (0-100)
    api_score = (has_api_access / total_datasets) * 100 if total_datasets > 0 else 0
    machine_readable_score = (has_machine_readable / total_datasets) * 100 if total_datasets > 0 else 0
    metadata_score = (has_metadata / total_datasets) * 100 if total_datasets > 0 else 0
    
    update_clarity = (sum(1 for u in update_indicators if u != 'unclear') / len(update_indicators)) * 100 if update_indicators else 0
    
    overall_score = (api_score + machine_readable_score + metadata_score + update_clarity) / 4
    
    assessment = {
        'provider_id': provider_id,
        'total_datasets': total_datasets,
        'scores': {
            'api_accessibility': round(api_score, 1),
            'machine_readability': round(machine_readable_score, 1),
            'metadata_quality': round(metadata_score, 1),
            'update_transparency': round(update_clarity, 1),
            'overall_governance': round(overall_score, 1)
        },
        'details': {
            'api_enabled_datasets': has_api_access,
            'machine_readable_datasets': has_machine_readable,
            'well_documented_datasets': has_metadata,
            'update_frequency_distribution': {
                'regular': update_indicators.count('regular'),
                'periodic': update_indicators.count('periodic'),
                'unclear': update_indicators.count('unclear')
            }
        },
        'governance_grade': get_governance_grade(overall_score)
    }
    
    return assessment

def get_governance_grade(score):
    """Convert numerical score to letter grade"""
    if score >= 85:
        return "A+ (Excellent)"
    elif score >= 75:
        return "A (Very Good)"
    elif score >= 65:
        return "B+ (Good)"
    elif score >= 55:
        return "B (Satisfactory)"
    elif score >= 45:
        return "C+ (Needs Improvement)"
    elif score >= 35:
        return "C (Poor)"
    else:
        return "D (Very Poor)"

def display_governance_assessment(assessment):
    """Display governance assessment in a readable format"""
    
    if not assessment:
        print("❌ No assessment available")
        return
    
    print(f"\n📊 DATA GOVERNANCE ASSESSMENT: {assessment['provider_id'].upper()}")
    print("=" * 70)
    
    print(f"📈 Overall Governance Grade: {assessment['governance_grade']}")
    print(f"📋 Total Datasets Analyzed: {assessment['total_datasets']}")
    
    print(f"\n🎯 GOVERNANCE METRICS:")
    scores = assessment['scores']
    print(f"├── API Accessibility: {scores['api_accessibility']}%")
    print(f"├── Machine Readability: {scores['machine_readability']}%")
    print(f"├── Metadata Quality: {scores['metadata_quality']}%")
    print(f"└── Update Transparency: {scores['update_transparency']}%")
    
    print(f"\n📊 DETAILED BREAKDOWN:")
    details = assessment['details']
    print(f"• {details['api_enabled_datasets']} datasets have API access")
    print(f"• {details['machine_readable_datasets']} datasets are machine-readable")
    print(f"• {details['well_documented_datasets']} datasets have good documentation")
    
    freq_dist = details['update_frequency_distribution']
    print(f"\n⏰ UPDATE FREQUENCY TRANSPARENCY:")
    print(f"• Regular updates: {freq_dist['regular']} datasets")
    print(f"• Periodic updates: {freq_dist['periodic']} datasets")
    print(f"• Unclear frequency: {freq_dist['unclear']} datasets")
    
    # Policy Recommendations
    print(f"\n💡 POLICY RECOMMENDATIONS:")
    if scores['api_accessibility'] < 50:
        print("🔧 Priority: Implement API access for more datasets")
    if scores['metadata_quality'] < 60:
        print("📝 Priority: Improve dataset documentation and metadata")
    if scores['update_transparency'] < 70:
        print("⏰ Priority: Clarify update schedules and data currency")
    
    if scores['overall_governance'] >= 75:
        print("✅ This department shows good data governance practices!")
    else:
        print("⚠️ This department has significant room for improvement in data governance")

# Example usage function
def complete_governance_analysis(provider_id):
    """
    Complete end-to-end governance analysis for a provider
    
    Args:
        provider_id: Government department code (e.g., 'hk-bd')
    
    Returns:
        dict: Complete analysis results
    """
    
    print(f"🔍 Starting complete governance analysis for: {provider_id}")
    print("=" * 60)
    
    # Step 1: Explore provider datasets
    print("Step 1: Exploring provider datasets...")
    dataset_results = explore_provider_datasets(provider_id)
    
    if not dataset_results or not dataset_results.get('datasets'):
        print(f"❌ Could not retrieve datasets for {provider_id}")
        return None
    
    print(f"✅ Found {len(dataset_results['datasets'])} datasets")
    
    # Step 2: Assess governance
    print("\nStep 2: Assessing data governance...")
    assessment = governance_assessment_framework(provider_id, dataset_results)
    
    # Step 3: Display results
    print("\nStep 3: Displaying governance assessment...")
    display_governance_assessment(assessment)
    
    # Step 4: Sample dataset access
    print(f"\nStep 4: Testing data access for sample datasets...")
    sample_datasets = dataset_results['datasets'][:3]  # Test first 3 datasets
    
    access_success = 0
    for i, dataset in enumerate(sample_datasets, 1):
        dataset_name = dataset.get('title', f'Dataset {i}')
        print(f"\n🔍 Testing access to: {dataset_name[:50]}...")
        
        # Try to access the dataset
        access_result = smart_dataset_access(dataset.get('link', ''))
        if access_result and access_result.get('success'):
            access_success += 1
            print(f"✅ Successfully accessed via {access_result.get('method', 'unknown')}")
        else:
            print(f"❌ Could not access dataset")
    
    success_rate = (access_success / len(sample_datasets)) * 100
    print(f"\n📊 Dataset Access Success Rate: {success_rate:.1f}%")
    
    return {
        'provider_id': provider_id,
        'dataset_count': len(dataset_results['datasets']),
        'governance_assessment': assessment,
        'access_success_rate': success_rate,
        'sample_datasets': sample_datasets
    }

print("🎯 DATA GOVERNANCE ASSESSMENT FRAMEWORK READY!")
print("\nTry these examples:")
print("• complete_governance_analysis('hk-bd')  # Buildings Department")
print("• complete_governance_analysis('hk-td')  # Transport Department") 
print("• complete_governance_analysis('hk-epd') # Environmental Protection")
print("• complete_governance_analysis('hk-censtatd') # Census & Statistics")

## Step 8: Your Project Assignment Framework

Based on your data.gov.hk exploration, here's how to structure your final project:

In [ ]:
def project_assignment_framework():
    """
    Complete framework for your data.gov.hk project assignment
    """
    
    print("🎯 YOUR DATA.GOV.HK PROJECT ASSIGNMENT FRAMEWORK")
    print("=" * 70)
    
    print("\n📋 PART 1: SYSTEMATIC EXPLORATION (Week 1-2)")
    print("1. Run: extract_data_providers() to see all government departments")
    print("2. Choose 3-5 providers from different policy areas")
    print("3. For each provider, run: explore_provider_datasets(provider_id)")
    print("4. Document your findings using the categories framework above")
    
    print("\n🔍 PART 2: GOVERNANCE ASSESSMENT (Week 3)")
    print("1. Select 2 providers for detailed governance analysis")
    print("2. Run: complete_governance_analysis(provider_id) for each")
    print("3. Compare governance scores and practices")
    print("4. Identify best practices and improvement areas")
    
    print("\n📊 PART 3: DEEP DATA ANALYSIS (Week 4-5)")
    print("1. Choose ONE specific dataset for detailed analysis")
    print("2. Use: smart_dataset_access() to download the data")
    print("3. Perform statistical analysis relevant to the policy area")
    print("4. Create visualizations showing key insights")
    
    print("\n📝 PART 4: POLICY RECOMMENDATIONS (Week 6)")
    print("1. Connect your data findings to governance assessment")
    print("2. Propose specific improvements to data practices")
    print("3. Suggest policy changes based on data evidence")
    print("4. Address data quality, accessibility, and transparency")
    
    print("\n🎤 DELIVERABLES:")
    print("• Jupyter notebook with complete analysis")
    print("• 15-minute presentation on findings")
    print("• Policy brief (2-3 pages) with recommendations")
    print("• Governance assessment report card")
    
    print("\n💡 SUCCESS CRITERIA:")
    print("✅ Demonstrates understanding of HK government data landscape")
    print("✅ Shows technical skills in data extraction and analysis")
    print("✅ Provides actionable governance recommendations")
    print("✅ Connects data quality to policy effectiveness")
    
    return True

def suggested_project_topics():
    """
    Specific project suggestions based on data availability and policy relevance
    """
    
    topics = {
        "🚌 Transport Efficiency Analysis": {
            "provider": "hk-td (Transport Department)",
            "focus": "Analyze bus route efficiency and propose optimization",
            "governance_angle": "Assess open data practices for transport planning",
            "example_question": "How does traffic data quality affect transport policy decisions?",
            "technical_skills": "Time series analysis, GIS mapping, efficiency metrics"
        },
        
        "🏠 Housing Policy Data Governance": {
            "provider": "hk-bd (Buildings Department) + hk-hd (Housing Department)",
            "focus": "Evaluate building safety data transparency",
            "governance_angle": "Compare data sharing between housing agencies",
            "example_question": "Does poor data governance contribute to housing policy delays?",
            "technical_skills": "Data integration, safety trend analysis, regulatory compliance tracking"
        },
        
        "🌱 Environmental Monitoring Effectiveness": {
            "provider": "hk-epd (Environmental Protection Department)",
            "focus": "Air quality data accuracy and public access",
            "governance_angle": "Assess real-time environmental data governance",
            "example_question": "How does environmental data quality affect public health policy?",
            "technical_skills": "Time series forecasting, correlation analysis, data quality metrics"
        },
        
        "💼 Economic Data Integration": {
            "provider": "hk-censtatd (Census and Statistics Department)",
            "focus": "Cross-departmental economic data consistency",
            "governance_angle": "Evaluate data standardization across agencies",
            "example_question": "How do data silos affect economic policy coordination?",
            "technical_skills": "Data harmonization, statistical modeling, trend analysis"
        },
        
        "🎓 Education Resource Allocation": {
            "provider": "hk-edb (Education Bureau)",
            "focus": "School performance data transparency and equity",
            "governance_angle": "Assess educational data privacy vs. transparency balance",
            "example_question": "Does education data governance support equitable resource allocation?",
            "technical_skills": "Equity analysis, resource optimization, performance metrics"
        }
    }
    
    print("🎯 SUGGESTED PROJECT TOPICS")
    print("=" * 60)
    
    for topic, details in topics.items():
        print(f"\n{topic}")
        print(f"🏛️ Provider: {details['provider']}")
        print(f"🔍 Focus: {details['focus']}")
        print(f"⚖️ Governance Angle: {details['governance_angle']}")
        print(f"❓ Key Question: {details['example_question']}")
        print(f"🛠️ Technical Skills: {details['technical_skills']}")
        print("-" * 50)
    
    print("\n💡 SELECTION STRATEGY:")
    print("1. Choose a topic that matches your policy interests")
    print("2. Ensure the provider has sufficient datasets available")
    print("3. Consider your technical comfort level")
    print("4. Think about real-world policy impact potential")
    
    return topics

# Run the framework
project_assignment_framework()
print("\n" + "="*70)
suggested_project_topics()

print("\n🚀 READY TO START YOUR PROJECT!")
print("Use the functions above to begin your systematic exploration of Hong Kong's government data landscape.")

# 🏛️ Phase 4: Policy Connection Framework

## Step 7: Linking Data to Government Decisions

In [ ]:
# Framework for connecting datasets to government policy
def policy_connection_analysis(dataset_name, data_summary):
    """
    Help identify policy connections for your chosen dataset
    """
    print(f"🏛️ POLICY CONNECTION ANALYSIS: {dataset_name}")
    print("=" * 60)
    
    # Policy areas framework
    policy_areas = {
        "🚌 Transport Policy": {
            "keywords": ["transport", "traffic", "bus", "mtr", "road", "parking", "vehicle"],
            "government_depts": ["Transport Department", "Highways Department"],
            "decisions": ["Route planning", "Traffic management", "Infrastructure investment"],
            "metrics": ["ridership", "accidents", "congestion", "emissions"]
        },
        "🌱 Environmental Policy": {
            "keywords": ["air", "pollution", "waste", "energy", "emission", "environment"],
            "government_depts": ["Environmental Protection Department", "Development Bureau"],
            "decisions": ["Pollution control", "Waste management", "Conservation measures"],
            "metrics": ["pollution index", "waste volume", "energy consumption"]
        },
        "🏥 Health Policy": {
            "keywords": ["health", "hospital", "disease", "medical", "clinic", "patient"],
            "government_depts": ["Department of Health", "Hospital Authority"],
            "decisions": ["Resource allocation", "Service planning", "Disease prevention"],
            "metrics": ["bed occupancy", "waiting times", "infection rates"]
        },
        "🏠 Housing Policy": {
            "keywords": ["housing", "property", "rent", "apartment", "building", "estate"],
            "government_depts": ["Housing Department", "Development Bureau"],
            "decisions": ["Public housing allocation", "Land use planning", "Rent control"],
            "metrics": ["housing prices", "waiting lists", "occupancy rates"]
        },
        "💼 Economic Policy": {
            "keywords": ["business", "employment", "income", "gdp", "economy", "trade"],
            "government_depts": ["Commerce and Economic Development Bureau", "Labour Department"],
            "decisions": ["Business regulation", "Employment support", "Economic development"],
            "metrics": ["unemployment rate", "business registrations", "economic indicators"]
        },
        "🎓 Education Policy": {
            "keywords": ["school", "student", "education", "teacher", "university", "learning"],
            "government_depts": ["Education Bureau"],
            "decisions": ["School funding", "Curriculum planning", "Resource allocation"],
            "metrics": ["enrollment", "performance", "teacher ratios"]
        }
    }
    
    # Analyze dataset for policy connections
    dataset_text = f"{dataset_name} {' '.join(data_summary.get('columns', []))}".lower()
    
    policy_matches = {}
    for policy_area, details in policy_areas.items():
        matches = 0
        matched_keywords = []
        
        for keyword in details["keywords"]:
            if keyword in dataset_text:
                matches += 1
                matched_keywords.append(keyword)
        
        if matches > 0:
            policy_matches[policy_area] = {
                'match_score': matches,
                'matched_keywords': matched_keywords,
                'details': details
            }
    
    # Display results
    if policy_matches:
        print("✅ POLICY CONNECTIONS IDENTIFIED:")
        sorted_matches = sorted(policy_matches.items(), key=lambda x: x[1]['match_score'], reverse=True)
        
        for policy_area, match_info in sorted_matches[:3]:  # Show top 3 matches
            print(f"\n{policy_area} (Score: {match_info['match_score']})")
            print(f"🔑 Keywords found: {', '.join(match_info['matched_keywords'])}")
            print(f"🏛️ Relevant departments: {', '.join(match_info['details']['government_depts'])}")
            print(f"📋 Potential decisions: {', '.join(match_info['details']['decisions'])}")
            print("-" * 40)
    
    else:
        print("⚠️ No clear policy connections identified automatically.")
        print("💡 Consider these general approaches:")
        print("• Look for regulatory compliance aspects")
        print("• Consider resource allocation decisions")
        print("• Examine service delivery efficiency")
        print("• Evaluate public interest implications")
    
    # Policy questions generator
    print("\n❓ SUGGESTED POLICY RESEARCH QUESTIONS:")
    if policy_matches:
        top_policy = sorted_matches[0]
        policy_name = top_policy[0].replace('🏛️', '').replace('🚌', '').replace('🌱', '').replace('🏥', '').replace('🏠', '').replace('💼', '').replace('🎓', '').strip()
        
        questions = [
            f"How can {policy_name.lower()} be optimized using data-driven insights?",
            f"What patterns in the data suggest improvements to current {policy_name.lower()}?",
            f"How do data governance practices affect {policy_name.lower()} effectiveness?",
            f"What additional data should government collect to improve {policy_name.lower()}?"
        ]
    else:
        questions = [
            "How can government data collection be improved for this domain?",
            "What data governance challenges are evident in this dataset?",
            "How could better data curation support policy decisions?",
            "What data quality improvements would enhance government decision-making?"
        ]
    
    for i, question in enumerate(questions, 1):
        print(f"{i}. {question}")
    
    return policy_matches

# Example usage
print("🎯 POLICY CONNECTION FRAMEWORK")
print("Use this to connect your dataset to government decisions:")
print("policy_analysis = policy_connection_analysis('Dataset Name', analysis_results)")

# 🔧 Phase 5: Data Governance Analysis Framework

## Step 8: Governance Assessment Template

In [ ]:
# Data governance assessment framework
def data_governance_assessment(dataset_info, dataset_url):
    """
    Assess data governance quality and generate improvement recommendations
    """
    print("🔧 DATA GOVERNANCE ASSESSMENT")
    print("=" * 50)
    
    # Governance criteria checklist
    governance_criteria = {
        "📊 Data Quality": {
            "completeness": "How complete is the dataset? (missing values, coverage)",
            "accuracy": "How accurate is the data? (errors, inconsistencies)",
            "timeliness": "How current is the data? (update frequency, lag time)",
            "consistency": "How consistent is the format? (standardization, conventions)"
        },
        "📋 Documentation": {
            "metadata": "Are variables clearly defined? (data dictionary available)",
            "methodology": "Is data collection method documented? (process transparency)",
            "context": "Is policy context provided? (purpose, use cases)",
            "limitations": "Are data limitations acknowledged? (known issues, scope)"
        },
        "🌐 Accessibility": {
            "format": "Is data in machine-readable format? (CSV, JSON vs PDF)",
            "availability": "Is data easily discoverable? (search, navigation)",
            "download": "Is download process straightforward? (no barriers)",
            "api": "Is API access available? (programmatic access)"
        },
        "🔄 Maintenance": {
            "updates": "How frequently is data updated? (regular schedule)",
            "versioning": "Is historical data preserved? (version control)",
            "contact": "Is there clear contact for questions? (support available)",
            "feedback": "Is there mechanism for user feedback? (improvement process)"
        }
    }
    
    assessment_results = {}
    total_score = 0
    max_score = 0
    
    # Interactive assessment
    print(f"Assessing dataset: {dataset_info.get('name', 'Unknown')}")
    print(f"URL: {dataset_url}")
    print("\nRate each aspect (1=Poor, 2=Fair, 3=Good, 4=Excellent, 0=Unknown):")
    print("=" * 60)
    
    for category, criteria in governance_criteria.items():
        print(f"\n{category}")
        category_scores = {}
        
        for criterion, description in criteria.items():
            print(f"  {criterion}: {description}")
            while True:
                try:
                    score = int(input(f"    Rate {criterion} (0-4): "))
                    if 0 <= score <= 4:
                        category_scores[criterion] = score
                        if score > 0:  # Only count towards total if not "Unknown"
                            total_score += score
                            max_score += 4
                        break
                    else:
                        print("    Please enter a number between 0-4")
                except ValueError:
                    print("    Please enter a valid number")
        
        assessment_results[category] = category_scores
    
    # Calculate overall governance score
    overall_score = (total_score / max_score * 100) if max_score > 0 else 0
    
    print(f"\n📊 GOVERNANCE ASSESSMENT RESULTS")
    print("=" * 40)
    print(f"Overall Governance Score: {overall_score:.1f}%")
    
    # Category breakdown
    for category, scores in assessment_results.items():
        category_avg = sum(s for s in scores.values() if s > 0) / len([s for s in scores.values() if s > 0]) if any(s > 0 for s in scores.values()) else 0
        print(f"{category}: {category_avg:.1f}/4.0")
    
    # Generate recommendations
    print(f"\n💡 IMPROVEMENT RECOMMENDATIONS")
    print("=" * 40)
    
    recommendations = []
    
    # Specific recommendations based on low scores
    for category, scores in assessment_results.items():
        for criterion, score in scores.items():
            if score == 1:  # Poor scores get specific recommendations
                if criterion == "completeness":
                    recommendations.append("🔧 Improve data completeness: Implement validation checks and mandatory field requirements")
                elif criterion == "metadata":
                    recommendations.append("📋 Create comprehensive data dictionary with variable definitions and units")
                elif criterion == "format":
                    recommendations.append("💾 Provide data in machine-readable formats (CSV, JSON) instead of PDF")
                elif criterion == "updates":
                    recommendations.append("🔄 Establish regular update schedule and communicate timing to users")
                elif criterion == "api":
                    recommendations.append("🌐 Develop API access for programmatic data retrieval")
    
    # General recommendations based on overall score
    if overall_score >= 80:
        recommendations.append("✅ Excellent governance practices - consider as best practice example")
    elif overall_score >= 60:
        recommendations.append("⚡ Good foundation - focus on specific improvement areas identified above")
    elif overall_score >= 40:
        recommendations.append("🔧 Significant improvements needed - prioritize data quality and documentation")
    else:
        recommendations.append("🚨 Major governance gaps - comprehensive reform needed")
    
    # Display recommendations
    if recommendations:
        for i, rec in enumerate(recommendations, 1):
            print(f"{i}. {rec}")
    
    return {
        'overall_score': overall_score,
        'category_results': assessment_results,
        'recommendations': recommendations
    }

print("🎯 DATA GOVERNANCE ASSESSMENT")
print("Use this to evaluate and improve data governance:")
print("governance_results = data_governance_assessment(dataset_info, dataset_url)")

# 🎯 Phase 6: Your Project Development

## Step 9: Project Checklist and Next Steps

In [ ]:
# Project development checklist
def project_checklist():
    """
    Interactive checklist for your open data exploration project
    """
    checklist_items = {
        "🔍 Discovery Phase": [
            "Browse data.gov.hk systematically by category",
            "Identify 3-5 potential datasets of interest", 
            "Download sample data for initial assessment",
            "Document dataset URLs and basic information"
        ],
        "📊 Dataset Selection": [
            "Use quick_dataset_preview() for each candidate",
            "Run comprehensive_dataset_analysis() on top choices",
            "Evaluate modeling potential (regression/simulation feasibility)",
            "Select ONE dataset for your project focus"
        ],
        "🏛️ Policy Connection": [
            "Run policy_connection_analysis() on chosen dataset",
            "Research relevant government departments and decisions",
            "Formulate specific policy research questions",
            "Connect analysis to real government decision-making"
        ],
        "🔧 Governance Assessment": [
            "Complete data_governance_assessment() evaluation",
            "Identify specific governance improvement opportunities",
            "Research best practices from other jurisdictions",
            "Develop actionable recommendations"
        ],
        "📈 Analysis Approach": [
            "Choose primary method: regression/simulation OR governance focus",
            "Set up analysis framework using provided templates",
            "Plan visualizations and key findings presentation",
            "Prepare policy implications and recommendations"
        ],
        "📋 Documentation": [
            "Document data discovery process and selection rationale",
            "Create comprehensive analysis notebook",
            "Prepare policy brief with recommendations",
            "Plan final presentation for class discussion"
        ]
    }
    
    print("✅ OPEN DATA EXPLORATION PROJECT CHECKLIST")
    print("=" * 60)
    print("Use this checklist to track your progress:\n")
    
    for phase, items in checklist_items.items():
        print(f"{phase}")
        for item in items:
            print(f"  ☐ {item}")
        print()
    
    print("🎯 SUCCESS CRITERIA:")
    print("✅ Clear policy relevance with government connection")
    print("✅ Appropriate analytical approach (regression/simulation OR governance)")
    print("✅ Actionable recommendations for policy improvements")
    print("✅ Evidence of systematic data exploration process")
    print("✅ Professional presentation of findings")
    
    print("\n💡 REMEMBER:")
    print("• Be patient - good datasets require systematic exploration")
    print("• Focus on ONE specific dataset with clear policy connection")
    print("• Choose approach based on data characteristics")
    print("• This is MORE EXCITING with some uncertainty - embrace the exploration!")
    
    return checklist_items

# Display the checklist
checklist = project_checklist()

# 🚀 Ready to Start Your Exploration!

## Quick Start Guide

1. **Start Exploring:** Browse data.gov.hk and use the functions above to analyze potential datasets
2. **Be Patient:** Remember, finding the right dataset takes time and systematic exploration
3. **Choose Your Approach:** 
   - **High modeling potential:** Focus on regression/simulation analysis
   - **Limited modeling potential:** Focus on data governance improvements
4. **Connect to Policy:** Always link your analysis to specific government decisions
5. **Document Everything:** Keep track of your exploration process

## Remember: This Topic is More Exciting with Some Uncertainty!

Unlike other topics with solid foundations, you're building something completely new. Embrace the exploration process and the uncertainty - that's what makes this option exciting and rewarding!

**Good luck with your data exploration journey! 🎯**